In [12]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['DEEPSEEK_API_KEY']=os.getenv('DEEPSEEK_API_KEY')
os.environ['DEEPSEEK_BASE_URL']=os.getenv('DEEPSEEK_BASE_URL')
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [13]:
from langchain_community.document_loaders import WebBaseLoader
loader=WebBaseLoader("https://docs.langchain.com/oss/python/langchain/install/")
loader


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [14]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langchain/install/', 'title': 'Install LangChain - Docs by LangChain', 'language': 'en'}, page_content='Install LangChain - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageOpen sourceSearch...⌘KGitHubTry LangSmithTry LangSmithSearch...NavigationGet startedInstall LangChainDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIDeploy with LangSmithDeploymentObservabilityGet startedInstall LangChain

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=20)
docuements=text_splitter.split_documents(docs)
docuements


[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langchain/install/', 'title': 'Install LangChain - Docs by LangChain', 'language': 'en'}, page_content='Install LangChain - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageOpen sourceSearch...⌘KGitHubTry LangSmithTry LangSmithSearch...NavigationGet startedInstall LangChainDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryStreamingStructured'),
 Document(metadata={'source': 'https://docs.langchain.com/oss/python/langchain/install/', 'title': 'Install LangChain - Docs by LangChain', 'language': 'en'}, page_content='outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP

In [17]:
#由于无法使用openai api，先在终端启动ollama服务进行词嵌入
from langchain_community.embeddings import OllamaEmbeddings
embeddings=(
    OllamaEmbeddings(model="gemma:2b")
)


C:\Users\YuXin\AppData\Local\Temp\ipykernel_13200\3943865649.py:4: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  OllamaEmbeddings(model="gemma:2b")


In [18]:
from langchain_community.vectorstores import FAISS
storedb=FAISS.from_documents(docs, embeddings)
storedb


In [19]:
query="什么是Langchain"
results=storedb.similarity_search(query)
results[0].page_content


'Install LangChain - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageOpen sourceSearch...⌘KGitHubTry LangSmithTry LangSmithSearch...NavigationGet startedInstall LangChainDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIDeploy with LangSmithDeploymentObservabilityGet startedInstall LangChainCopy pageCopy pageTo install the LangChain package:\npipuvpip install -U langchain\n# Requires Python 3.10+\n\nLangChain provides integrations to hundreds of LLMs and thous

In [20]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="deepseek-chat",
    openai_api_key="sk-8b3ca69de6e849b49277d23112d9efab",
    openai_api_base="https://api.deepseek.com",
)
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000002C7C90A5590>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002C7C90A60D0>, root_client=<openai.OpenAI object at 0x000002C7C90A4CD0>, root_async_client=<openai.AsyncOpenAI object at 0x000002C7C90A5E50>, model_name='deepseek-chat', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.deepseek.com')

In [21]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
prompt=ChatPromptTemplate.from_template( """
根据以下上下文回答问题：
<context>
{context}
</context>
问题: {input}
"""
)
document_chain = create_stuff_documents_chain(llm, prompt)




In [22]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"what is langchain?",
    "context":[Document(page_content="Langchain是一个基于Python的框架,用于构建智能体和应用.")]
    
    
 })



'根据提供的上下文，Langchain是一个基于Python的框架，用于构建智能体（agents）和应用（applications）。'

In [26]:
retriver=storedb.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieve_chain=create_retrieval_chain(retriver, document_chain)
response=retrieve_chain.invoke({"input":"什么是Langchain"})
response["answer"]


'基于提供的上下文，LangChain 是一个开源框架（库），用于开发由大型语言模型（LLM）驱动的应用程序。\n\n具体信息如下：\n\n1.  **核心功能**：它是一个“Python”包（`langchain`），可以通过 `pip install -U langchain` 命令安装。\n2.  **集成能力**：LangChain 提供了与数百个大型语言模型（如 OpenAI、Anthropic）以及数千个其他第三方服务的集成。这些集成作为独立的提供商包存在，例如 `langchain-openai` 或 `langchain-anthropic`。\n3.  **应用方向**：上下文提到 LangChain 涉及代理（Agents）、模型（Models）、消息（Messages）、工具（Tools）、记忆（Memory）、流式输出（Streaming）、结构化输出（Structured output）等核心组件，以及多代理（Multi-agent）、检索（Retrieval）和可观测性（Observability）等高级功能。'